# Тестирование задач 9 и 10

Этот ноутбук тестирует реализацию анализа стабильности во времени границы эффективных портфелей.

**Задача 9**: Продемонстрировать динамику изменения границы эффективных портфелей (без ограничений) для скользящего и расширяющегося окон.

**Задача 10**: То же самое с экспоненциальным забыванием.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from task_9_10 import (
    calculate_efficient_frontier,
    calculate_efficient_frontier_weights,
    calculate_efficient_frontiers_over_time,
    analyze_efficient_frontier_stability,
    analyze_portfolio_composition_stability,
    efficient_frontier_dynamics_rolling,
    task_9b_efficient_frontier_dynamics_expanding,
    task_10_efficient_frontier_dynamics_exponential,
    task_10_exp_efficient_frontier_dynamics_expanding,
    compare_frontier_methods
)

# Настройка графиков
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

## 1. Загрузка данных

In [ ]:
# Импорт функций загрузки из task_2_3
sys.path.insert(0, '..')
from task_2_3 import load_prices_data, calculate_returns

# Загрузка данных
prices = load_prices_data('../data/prices_moex_new.csv')

print("=== Информация о данных ===")
print(f"Размер: {prices.shape}")
print(f"Период: {prices.index.min()} - {prices.index.max()}")
print(f"Количество акций: {len(prices.columns)}")
print(f"\nСписок акций: {list(prices.columns)}")
print("\nПервые 5 строк:")
print(prices.head())

## 2. Расчет доходностей

In [ ]:
# Расчет логарифмических доходностей
returns = calculate_returns(prices)

print("=== Информация о доходностях ===")
print(f"Размер: {returns.shape}")
print(f"Период: {returns.index.min()} - {returns.index.max()}")
print(f"\nОписательная статистика доходностей:")
print(returns.describe())

## 3. Тестирование функции эффективной границы

In [ ]:
# Проверка функции эффективной границы на простом примере
print("=== Тестирование функции эффективной границы ===")

# Используем данные за первый год
first_year_returns = returns.iloc[:251]
mean_returns = first_year_returns.mean().values
cov_matrix = first_year_returns.cov().values

print(f"Размер: {len(mean_returns)} активов")
print(f"Средние доходности (первые 5): {mean_returns[:5]}")

# Расчет эффективной границы
ef_returns, ef_stds = calculate_efficient_frontier(mean_returns, cov_matrix, n_points=50)

print(f"\nЭффективная граница: {len(ef_returns)} точек")
print(f"Мин. доходность: {ef_returns[0]:.6f}, Стд: {ef_stds[0]:.6f}")
print(f"Макс. доходность: {ef_returns[-1]:.6f}, Стд: {ef_stds[-1]:.6f}")

# Визуализация
plt.figure(figsize=(10, 6))
plt.plot(ef_stds, ef_returns, 'b-', linewidth=2, label='Эффективная граница')
plt.scatter(ef_stds, ef_returns, c=ef_returns, cmap='viridis', s=20)
plt.colorbar(label='Доходность')
plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Доходность')
plt.title('Эффективная граница (без ограничений)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Задача 9a: Скользящее окно

In [ ]:
print("=== Задача 9a: Анализ эффективной границы скользящим окном ===")
print("Окно: 1 год, Шаг: 1 год")

rolling_frontiers, rolling_stability = efficient_frontier_dynamics_rolling(
    returns, window_size='1Y', step_size='1Y', n_points=50
)

print(f"\nПолучено окон: {len(rolling_frontiers)}")
print(f"\nМетрики стабильности:")
print(rolling_stability)

In [ ]:
# Визуализация эффективных границ для всех окон
plt.figure(figsize=(14, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(rolling_frontiers)))

for i, (date, frontier) in enumerate(sorted(rolling_frontiers.items())):
    label = date.strftime('%Y-%m-%d') if i % 2 == 0 or i == len(rolling_frontiers) - 1 else None
    plt.plot(frontier['stds'], frontier['returns'], 
             color=colors[i], linewidth=2, alpha=0.7, label=label)

plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Доходность')
plt.title('Динамика эффективных границ (скользящее окно 1 год)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# График динамики минимального риска во времени
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(rolling_stability.index, rolling_stability['min_std'], marker='o', linewidth=2)
plt.title('Минимальный риск (GMVP) во времени')
plt.xlabel('Дата')
plt.ylabel('Стандартное отклонение')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(rolling_stability.index, rolling_stability['min_std_return'], marker='s', linewidth=2, color='orange')
plt.title('Доходность GMVP во времени')
plt.xlabel('Дата')
plt.ylabel('Доходность')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# График динамики максимального Шарпа во времени
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(rolling_stability.index, rolling_stability['max_sharpe'], marker='o', linewidth=2)
plt.title('Максимальное Шарп-отношение во времени')
plt.xlabel('Дата')
plt.ylabel('Шарп-отношение')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(rolling_stability.index, rolling_stability['efficiency_ratio'], marker='s', linewidth=2, color='green')
plt.title('Коэффициент эффективности (max_return / min_std) во времени')
plt.xlabel('Дата')
plt.ylabel('Коэффициент эффективности')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 5. Задача 9b: Расширяющееся окно

In [ ]:
print("=== Задача 9b: Анализ эффективной границы расширяющимся окном ===")
print("Шаг: 1 год")

expanding_frontiers, expanding_stability = task_9b_efficient_frontier_dynamics_expanding(
    returns, step_size='1Y', n_points=50
)

print(f"\nПолучено окон: {len(expanding_frontiers)}")
print(f"\nМетрики стабильности:")
print(expanding_stability)

In [ ]:
# Визуализация эффективных границ для расширяющегося окна
plt.figure(figsize=(14, 8))
colors = plt.cm.plasma(np.linspace(0, 1, len(expanding_frontiers)))

for i, (date, frontier) in enumerate(sorted(expanding_frontiers.items())):
    label = date.strftime('%Y-%m-%d') if i % 2 == 0 or i == len(expanding_frontiers) - 1 else None
    plt.plot(frontier['stds'], frontier['returns'], 
             color=colors[i], linewidth=2, alpha=0.7, label=label)

plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Доходность')
plt.title('Динамика эффективных границ (расширяющееся окно)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Задача 10: Экспоненциальное забывание (скользящее окно)

In [ ]:
print("=== Задача 10: Анализ эффективной границы с экспоненциальным забыванием ===")
print("Скользящее окно (1 год), λ=0.94")

rolling_exp_frontiers, rolling_exp_stability = task_10_efficient_frontier_dynamics_exponential(
    returns, window_size='1Y', step_size='1Y', lambda_param=0.94, n_points=50
)

print(f"\nПолучено окон: {len(rolling_exp_frontiers)}")
print(f"\nМетрики стабильности:")
print(rolling_exp_stability)

In [ ]:
# Визуализация эффективных границ с экспоненциальным забыванием
plt.figure(figsize=(14, 8))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(rolling_exp_frontiers)))

for i, (date, frontier) in enumerate(sorted(rolling_exp_frontiers.items())):
    label = date.strftime('%Y-%m-%d') if i % 2 == 0 or i == len(rolling_exp_frontiers) - 1 else None
    plt.plot(frontier['stds'], frontier['returns'], 
             color=colors[i], linewidth=2, alpha=0.7, label=label)

plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Доходность')
plt.title('Динамика эффективных границ (скользящее окно, эксп. забывание)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Сравнение методов

In [ ]:
# Сравнение всех трех методов
comparison = compare_frontier_methods([
    ('Скользящее окно', rolling_frontiers),
    ('Расширяющееся окно', expanding_frontiers),
    ('Скользящее окно (эксп. забывание)', rolling_exp_frontiers)
])

print("=== Сравнение методов ===")
print(comparison)

In [ ]:
# Визуальное сравнение минимального риска
plt.figure(figsize=(14, 6))

for method in comparison.index.get_level_values(0).unique():
    data = comparison.loc[method]
    plt.plot(data.index, data['min_std'], marker='o', linewidth=2, label=method)

plt.title('Сравнение минимального риска по методам')
plt.xlabel('Дата')
plt.ylabel('Стандартное отклонение')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Визуальное сравнение максимального Шарпа
plt.figure(figsize=(14, 6))

for method in comparison.index.get_level_values(0).unique():
    data = comparison.loc[method]
    plt.plot(data.index, data['max_sharpe'], marker='s', linewidth=2, label=method)

plt.title('Сравнение максимального Шарп-отношения по методам')
plt.xlabel('Дата')
plt.ylabel('Шарп-отношение')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Анализ стабильности состава портфелей

In [ ]:
# Анализ состава портфелей для скользящего окна
print("=== Анализ стабильности состава портфелей ===")

# Анализ для портфеля с 50-м перцентилем (медиана)
portfolio_stability = analyze_portfolio_composition_stability(
    rolling_frontiers, returns.columns, percentile=50
)

print(f"\nСостав портфелей во времени (медианный портфель):")
print(portfolio_stability[[f'w_{col}' for col in returns.columns[:5]] + ['portfolio_return', 'portfolio_std']])

In [ ]:
# Визуализация динамики весов для первых 5 акций
plt.figure(figsize=(14, 8))

for i, ticker in enumerate(returns.columns[:5]):
    plt.plot(portfolio_stability.index, portfolio_stability[f'w_{ticker}'], 
             marker='o', linewidth=2, label=ticker)

plt.title('Динамика весов портфеля (медианный портфель, скользящее окно)')
plt.xlabel('Дата')
plt.ylabel('Вес в портфеле')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# График суммарного изменения весов
plt.figure(figsize=(12, 6))
plt.bar(portfolio_stability.index, portfolio_stability['total_weight_change'].fillna(0))
plt.title('Суммарное изменение весов между периодами')
plt.xlabel('Дата')
plt.ylabel('Суммарное изменение весов')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Итоговый анализ

In [ ]:
print("=== Итоговый анализ ===")
print(f"\nПериод данных: {returns.index.min()} - {returns.index.max()}")
print(f"Всего наблюдений: {len(returns)}")
print(f"Количество акций: {len(returns.columns)}")

print(f"\n--- Скользящее окно (Задача 9a) ---")
print(f"Количество окон: {len(rolling_frontiers)}")
print(f"Средний минимальный риск: {rolling_stability['min_std'].mean():.6f}")
print(f"Средний макс. Шарп: {rolling_stability['max_sharpe'].mean():.6f}")

print(f"\n--- Расширяющееся окно (Задача 9b) ---")
print(f"Количество окон: {len(expanding_frontiers)}")
print(f"Средний минимальный риск: {expanding_stability['min_std'].mean():.6f}")
print(f"Средний макс. Шарп: {expanding_stability['max_sharpe'].mean():.6f}")

print(f"\n--- Экспоненциальное забывание (Задача 10) ---")
print(f"Количество окон: {len(rolling_exp_frontiers)}")
print(f"Средний минимальный риск: {rolling_exp_stability['min_std'].mean():.6f}")
print(f"Средний макс. Шарп: {rolling_exp_stability['max_sharpe'].mean():.6f}")

## Вывод

В этом ноутбуке мы протестировали:
1. Реализацию функции построения эффективной границы без ограничений
2. Анализ динамики эффективных границ скользящим окном (Задача 9a)
3. Анализ динамики эффективных границ расширяющимся окном (Задача 9b)
4. Анализ динамики эффективных границ с экспоненциальным забыванием (Задача 10)
5. Сравнение различных методов построения эффективных границ
6. Анализ стабильности состава портфелей во времени

Все функции работают корректно и позволяют анализировать стабильность эффективной границы во времени.